# Session 2 lab · Image formation & sensing

The code behind the slides. Each cell shows **one idea** in a few lines:

1. run it and look at the result;
2. change a number marked **`· try`** (the values after it are good ones to try) and run it again;
3. read *What to notice* under the cell.

| Part | Ideas |
|---|---|
| **A · Pinhole** | how big things look, a cube in perspective, parallel rails that meet |
| **B · Lenses** | field of view, zoom = crop, aperture and blur |
| **C · Sensors** | the image is an array, resolution, bit depth, shot noise, dynamic range |
| **D · Colour** | three channels, the Bayer mosaic, demosaicing, turning hues |

Nothing to hand in, nothing to upload — it runs on free Colab.

In [ ]:
# Setup — run this cell first
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data

photo = data.astronaut()                          # a 512 × 512 colour photo (NASA, public domain)
gray = cv2.cvtColor(photo, cv2.COLOR_RGB2GRAY)    # the same photo in grey
rng = np.random.default_rng(0)                    # random numbers for the noise cell
print('ready')

## A · Pinhole & perspective

A pinhole sends the point $(X, Y, Z)$ to $x = f\,X/Z,\; y = f\,Y/Z$ on the image
($f$ in mm, $X, Y, Z$ in m — the metres cancel, so $x, y$ are in mm).

In [ ]:
# A1 · How big on the image?  (the distance slider in the slides)
f = 50          # focal length (mm) · try 24, 200
height = 1.8    # the person's height (m)
Z = 10          # distance to the person (m) · try 20, 40

x = f * height / Z
print(f'image height = {f} mm × {height} m ÷ {Z} m = {x:.2f} mm')

**What to notice:** double `Z` and the image height halves; double `f` and it doubles.

In [ ]:
# A2 · A 1 m cube in perspective
f = 50          # focal length (mm) · try 24, 100
Z = 4           # distance to the front of the cube (m) · try 8, 16
shift = 1       # how far right of centre the cube sits (m) · try 0, 3

# the 8 corners: four at the front (distance Z), four at the back (Z + 1)
X = np.array([0, 1, 1, 0, 0, 1, 1, 0]) + shift
Y = np.array([0, 0, 1, 1, 0, 0, 1, 1]) - 0.5
D = np.array([Z, Z, Z, Z, Z + 1, Z + 1, Z + 1, Z + 1])

x = f * X / D           # the pinhole formula, for all 8 corners at once
y = f * Y / D

plt.figure(figsize=(4.5, 4.5))
plt.plot(x[[0, 1, 2, 3, 0]], y[[0, 1, 2, 3, 0]], color='#855400', label='front face')
plt.plot(x[[4, 5, 6, 7, 4]], y[[4, 5, 6, 7, 4]], color='coral', label='back face')
for i in range(4):
    plt.plot([x[i], x[i + 4]], [y[i], y[i + 4]], color='gray')
plt.xlim(-30, 30); plt.ylim(-30, 30); plt.gca().set_aspect('equal')
plt.xlabel('x on the image (mm)'); plt.legend(); plt.title(f'1 m cube at Z = {Z} m, f = {f} mm')
plt.show()
print(f'front face {x[1] - x[0]:.2f} mm wide · back face {x[5] - x[4]:.2f} mm wide')

**What to notice:** the back face is smaller only because it is farther away — that difference *is* perspective. Move the cube away (`Z = 16`) and the two faces become almost the same size.

In [ ]:
# A3 · Parallel rails meet at a vanishing point
f = 20                # focal length (mm) · try 50
gauge = 1.5           # distance between the two rails (m) · try 3
camera_height = 1.5   # camera above the rails (m)

Z = np.linspace(2, 300, 500)          # points along the rails, from 2 m to 300 m ahead
for X in (-gauge / 2, gauge / 2):     # left rail, right rail
    plt.plot(f * X / Z, f * -camera_height / Z, color='#444')
plt.plot(0, 0, 'o', color='coral', label='vanishing point')
plt.gca().set_aspect('equal'); plt.legend(); plt.title('two parallel rails through a pinhole')
plt.xlabel('x (mm)'); plt.ylabel('y (mm)'); plt.show()

for d in (10, 100, 1000):
    print(f'gap between the rails at {d:4d} m: {f * gauge / d:.3f} mm on the image')

**What to notice:** the gap on the image is $f \cdot \text{gauge} / Z$ — it never reaches 0, but it heads for the same point whatever `gauge` or `f` you choose.

## B · Lenses

A lens gathers many rays into one point, so the picture is bright **and** sharp — but only at one distance.

In [ ]:
# B1 · Field of view: FOV = 2 · arctan(w / 2f)
w = 36          # sensor width (mm) — 36 = full frame · try 24, 6.4
f = 50          # focal length (mm) · try 18, 200

fov = 2 * np.degrees(np.arctan(w / (2 * f)))
print(f'f = {f} mm on a {w} mm sensor → field of view {fov:.1f}°')

**What to notice:** longer `f` → narrower view. A smaller sensor (`w = 6.4`, a phone) sees less through the same `f`.

In [ ]:
# B2 · Zoom = keep the centre. Pretend the photo was taken at 24 mm.
f0 = 24         # focal length of the photo (mm)
f = 50          # focal length to imitate (mm) · try 35, 100

keep = f0 / f                         # a longer lens sees this fraction of the width
h, w = gray.shape
ch, cw = int(h * keep), int(w * keep)
top, left = (h - ch) // 2, (w - cw) // 2
crop = photo[top:top + ch, left:left + cw]

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(photo); ax[0].set_title(f'{f0} mm (the photo)')
ax[1].imshow(crop);  ax[1].set_title(f'{f} mm = the centre {keep:.0%}')
for a in ax: a.axis('off')
plt.show()
print(f'the crop is {cw} × {ch} pixels out of {w} × {h}')

**What to notice:** cropping looks like a longer lens, but it throws pixels away — try `f = 100` and count what is left.

In [ ]:
# B3 · Aperture: brightness and blur (thin lens)
f = 50          # focal length (mm)
N = 2.8         # f-number; the opening is f / N wide · try 1.8, 16
focus = 2       # the lens is focused at this distance (m) · try 6

A = f / N                                   # opening (mm)
s = focus * 1000                            # focus distance in mm
z = np.linspace(0.5, 10, 300) * 1000        # object distances 0.5–10 m, in mm
blur = A * f * np.abs(z - s) / (z * (s - f))     # size of the blur circle on the sensor (mm)

plt.plot(z / 1000, blur)
plt.axvline(focus, color='gray', ls=':', label='in focus')
plt.ylim(0, 2); plt.xlabel('object distance (m)'); plt.ylabel('blur circle (mm)')
plt.legend(); plt.title(f'f = {f} mm at f/{N}'); plt.show()

blur_1m = A * f * abs(1000 - s) / (1000 * (s - f))
print(f'opening {A:.1f} mm · light compared with f/1.8: ×{(1.8 / N) ** 2:.3f} · blur at 1 m: {blur_1m:.2f} mm')

**What to notice:** a smaller opening (`N = 16`) flattens the curve — more of the scene is sharp — but lets in far less light. That is the trade on the depth-of-field slide.

## C · Sensors

To the computer an image is an array of numbers: height × width (× 3 for colour), each 0–255 in 8 bits.

In [ ]:
# C1 · The image is an array
print('shape:', gray.shape, '· type:', gray.dtype, '· smallest:', gray.min(), '· largest:', gray.max())
print(gray[100:105, 200:205])         # a 5 × 5 patch — just numbers

In [ ]:
# C2 · Resolution
n = 32          # keep n × n pixels · try 128, 16

small = cv2.resize(gray, (n, n), interpolation=cv2.INTER_AREA)    # each new pixel = average of a block
plt.imshow(small, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
plt.title(f'{n} × {n} pixels'); plt.axis('off'); plt.show()

**What to notice:** at 16 × 16 you still see a person, but not a face — resolution sets the smallest detail you can tell apart.

In [ ]:
# C3 · Bit depth: b bits keep 2**b grey levels
bits = 3        # bits per pixel · try 8, 1

levels = 2 ** bits
step = 256 / levels
fewer = (gray // step) * step         # round every value down to one of the levels

plt.imshow(fewer, cmap='gray', vmin=0, vmax=255)
plt.title(f'{bits} bits = {levels} grey levels'); plt.axis('off'); plt.show()

**What to notice:** with few levels, smooth areas (the sky, the cheek) break into bands; edges survive much longer.

In [ ]:
# C4 · Shot noise: a pixel that expects N photons gets a random (Poisson) number of them
photons = 64    # photons a white pixel collects · try 4, 1024

expected = photons * (gray / 255)         # photons each pixel should get on average
                                          # (gray / 255 first: gray holds 8-bit numbers, photons × gray would overflow)
counts = rng.poisson(expected)            # what it actually gets
noisy = counts / photons * 255            # back to 0–255

plt.imshow(noisy, cmap='gray', vmin=0, vmax=255)
plt.title(f'{photons} photons for white'); plt.axis('off'); plt.show()

patch = rng.poisson(photons, 100_000)     # a flat white patch
print(f'SNR = mean ÷ spread = {patch.mean() / patch.std():.1f}   ·   √photons = {np.sqrt(photons):.1f}')

**What to notice:** SNR follows √photons — 4 × the light gives only 2 × the SNR. Dark scenes are noisy because few photons arrive, not because the sensor is bad.

In [ ]:
# C5 · Dynamic range: a dark room with a bright window is 1000 : 1, but 8 bits hold 255 : 1
exposure = 0    # stops; +1 doubles the light · try -6, 2

scene = np.tile(np.linspace(2, 20, 320), (200, 1))   # the room, darker on the left
scene[30:120, 190:290] = 1000                         # the window
pixel = scene * 2.0 ** exposure * 255 / 40            # what the sensor would report
image = np.clip(pixel, 0, 255)                        # 8 bits keep only 0–255

plt.imshow(image, cmap='gray', vmin=0, vmax=255)
plt.title(f'exposure {exposure:+d}'); plt.axis('off'); plt.show()
print(f'pure white {np.mean(pixel >= 255):.0%} of the pixels · pure black {np.mean(pixel < 1):.0%}')

**What to notice:** no single `exposure` keeps both the window and the room — one end always clips. HDR merges several exposures.

## D · Colour

A colour pixel is three numbers (R, G, B). A single-chip sensor measures only **one** per photosite, through a Bayer filter that repeats **R G / G B**.

In [ ]:
# D1 · Three channels = three grey images
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
ax[0].imshow(photo); ax[0].set_title('colour')
for i, name in enumerate(['R', 'G', 'B']):
    ax[i + 1].imshow(photo[:, :, i], cmap='gray'); ax[i + 1].set_title(name)
for a in ax: a.axis('off')
plt.show()

**What to notice:** the red suit is bright in R and dark in G and B; the white parts are bright in all three.

In [ ]:
# D2 · What the sensor records: one number per photosite, in the pattern  R G / G B
size = 32       # side of the enlarged corner (pixels) · try 16, 128

h, w, _ = photo.shape
raw = np.zeros((h, w), np.uint8)
raw[0::2, 0::2] = photo[0::2, 0::2, 0]     # R: even rows, even columns
raw[0::2, 1::2] = photo[0::2, 1::2, 1]     # G: even rows, odd columns
raw[1::2, 0::2] = photo[1::2, 0::2, 1]     # G: odd rows, even columns
raw[1::2, 1::2] = photo[1::2, 1::2, 2]     # B: odd rows, odd columns

view = np.zeros_like(photo)                # paint each number in the colour of its filter
view[0::2, 0::2, 0] = raw[0::2, 0::2]
view[0::2, 1::2, 1] = raw[0::2, 1::2]
view[1::2, 0::2, 1] = raw[1::2, 0::2]
view[1::2, 1::2, 2] = raw[1::2, 1::2]

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(photo[330:330 + size, 150:150 + size], interpolation='nearest'); ax[0].set_title('what the scene looks like')
ax[1].imshow(view[330:330 + size, 150:150 + size], interpolation='nearest');  ax[1].set_title('what the sensor records')
for a in ax: a.axis('off')
plt.show()
print('photo:', photo.shape, '→ mosaic:', raw.shape, '— one number per photosite, not three')

**What to notice:** green gets twice as many photosites as red or blue — our eyes are most sensitive to green.

In [ ]:
# D3 · Demosaicing: fill in the two missing colours from the neighbours
size = 64       # side of the enlarged corner (pixels) · try 32, 256

rebuilt = cv2.cvtColor(raw, cv2.COLOR_BayerBG2RGB)
# OpenCV names the pattern after the 2 × 2 block that starts at row 1, column 1,
# so our R G / G B mosaic is 'BG' to OpenCV.

error = np.abs(rebuilt.astype(int) - photo.astype(int)).mean()
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(photo[150:150 + size, 220:220 + size]);   ax[0].set_title('original')
ax[1].imshow(rebuilt[150:150 + size, 220:220 + size]); ax[1].set_title('rebuilt from the mosaic')
for a in ax: a.axis('off')
plt.show()
print(f'average error: {error:.2f} out of 255')

**What to notice:** the rebuilt photo is close almost everywhere; the mistakes sit on sharp edges, where neighbours disagree.

In [ ]:
# D4 · Turn every hue (HSV); OpenCV stores hue as 0–179 = degrees ÷ 2
turn = 120      # degrees · try 60, 180

hsv = cv2.cvtColor(photo, cv2.COLOR_RGB2HSV)
hsv[:, :, 0] = (hsv[:, :, 0].astype(int) + turn // 2) % 180
turned = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(photo);  ax[0].set_title('original')
ax[1].imshow(turned); ax[1].set_title(f'hue + {turn}°')
for a in ax: a.axis('off')
plt.show()

**What to notice:** colours change but brightness and edges stay — hue is separate from value in HSV.

---
Back to the slides: [session 2 deck](https://classes.incortx.com/ComputerVision/session-02/slides.html) · exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s2).